[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C40_Research_Methodology_Course/01_reading_reproducing/01_reading_reproducing.ipynb)

# 01 · 读论文与复现（动手做）

目标：把读论文的核心动作变成可操作、可验证的代码——**拆主张-证据**、**读消融表做归因**、**从结果表复现一个数字**、**给可信度打分**。

路线：复现一个报告的准确率 → 复现一个易算错的指标(macro-F1) → 拆解消融表做归因 → 可信度打分器 → ✏️ 练习 → 📖 答案 → 🧪 真实论文数字胶囊。

> 核心心法：**复现的单位是一个具体数字**。与其说「我大概复现了这篇论文」，不如说「我复现了表 2 的 92.1%，误差 0.0%」。

## 1 · worked：从公开预测复现报告的准确率

场景：一篇论文报告「方法 X 测试准确率 = 92.0%」，并（按好实践）公开了每个测试样本的预测与真值。
我们**不重训模型**，只从公开预测**重新算一遍**那个数字，验证它确实能从公开材料复现。

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(0)

# 内置一份『论文公开的预测』：1000 个测试样本的 真值 与 模型预测
n = 1000
y_true = rng.integers(0, 3, size=n)                 # 3 分类
y_pred = y_true.copy()
wrong_idx = rng.choice(n, size=80, replace=False)   # 故意答错 80 个 -> 准确率应 = 92.0%
y_pred[wrong_idx] = (y_true[wrong_idx] + 1) % 3
paper = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred})
REPORTED_ACC = 0.920                                # 论文报告的数字（主张）

# 复现：用论文声明的指标定义（top-1 accuracy）重新算
my_acc = (paper['y_pred'] == paper['y_true']).mean()
print(f'论文报告准确率 = {REPORTED_ACC:.3f}')
print(f'我从公开预测复现 = {my_acc:.3f}')
print(f'差异 = {abs(my_acc - REPORTED_ACC):.4f}')
assert abs(my_acc - REPORTED_ACC) < 1e-9, '应当精确复现报告的准确率'
print('✅ 数字对得上 —— 这个准确率可从公开材料复现。')

## 2 · worked：同名指标，不同定义——复现失败的常见元凶

「F1」不是一个数，而是一**族**。`macro-F1`（各类 F1 求平均）与 `micro-F1`（全局汇总）在类别不平衡时**差很多**。
论文只写「F1=...」不写哪种，是复现对不上的经典原因。我们两种都算，看差多少。

In [ ]:
def precision_recall_f1(y_true, y_pred, cls):
    tp = np.sum((y_pred == cls) & (y_true == cls))
    fp = np.sum((y_pred == cls) & (y_true != cls))
    fn = np.sum((y_pred != cls) & (y_true == cls))
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return prec, rec, f1

def macro_f1(y_true, y_pred):
    classes = np.unique(y_true)
    return np.mean([precision_recall_f1(y_true, y_pred, c)[2] for c in classes])

def micro_f1(y_true, y_pred):
    # 多分类下 micro-F1 == accuracy（每个样本恰好一个预测、一个真值）
    tp = np.sum(y_pred == y_true)
    return tp / len(y_true)

# 造一个类别不平衡 + 小类表现差 的场景，放大 macro/micro 差异
yt = np.array([0]*900 + [1]*80 + [2]*20)
yp = yt.copy()
yp[900:980] = 0      # 类1 全预测错成 0
yp[980:]    = 0      # 类2 全预测错成 0
ma, mi = macro_f1(yt, yp), micro_f1(yt, yp)
print(f'macro-F1 = {ma:.3f}   (小类拖累，分数低)')
print(f'micro-F1 = {mi:.3f}   (被大类主导，分数高 = accuracy)')
print(f'两者相差 = {mi - ma:.3f}  <- 论文不写哪种，你就复现不出！')
assert mi - ma > 0.3, '不平衡时 micro 应远高于 macro'
print('✅ 教训：复现前必须搞清指标的【确切定义】，否则数字永远对不上。')

## 3 · worked：读消融表做贡献归因

一篇论文的方法 = 基线 + 3 个组件(A/B/C)。消融表给出去掉每个组件后的准确率。
我们算每个组件的**边际贡献**，找出**名不副实**（被宣传但其实没用）的组件。

In [ ]:
# 内置（仿真）消融表：full = 全部组件；-X = 去掉组件 X
ablation = pd.DataFrame({
    'config':   ['full', '-A', '-B', '-C', 'baseline(无ABC)'],
    'accuracy': [0.910,  0.882, 0.905, 0.860, 0.840],
})
print(ablation.to_string(index=False))

full = ablation.loc[ablation['config'] == 'full', 'accuracy'].iloc[0]
# 每个组件的贡献 ≈ 去掉它掉了多少分 = full - acc(-X)
contrib = {}
for comp in ['A', 'B', 'C']:
    acc_without = ablation.loc[ablation['config'] == f'-{comp}', 'accuracy'].iloc[0]
    contrib[comp] = full - acc_without
print('\n各组件边际贡献（去掉它掉的分）：')
for comp, v in sorted(contrib.items(), key=lambda kv: -kv[1]):
    print(f'  组件 {comp}: {v:+.3f}')

# 找名不副实的组件：贡献接近 0（这里设阈值 0.01）
useless = [comp for comp, v in contrib.items() if v < 0.01]
print(f'\n名不副实（贡献<0.01）的组件：{useless}')
assert contrib['C'] > contrib['A'] > contrib['B'], 'C 贡献最大、B 最小'
assert useless == ['B'], 'B 去掉几乎不掉分 = 名不副实'
print('✅ 归因结论：真正起作用的是 C 和 A；组件 B 几乎没贡献，论文若大肆宣传 B 就是名实不符。')

## 4 · worked：消融里藏着的混杂——把数据/算力的功劳记到方法头上

论文宣称「新模块带来 +2 分」。但若它**顺手**把训练数据翻倍了，那 2 分可能来自数据而非模块。
我们分解：总提升中，多少来自模块、多少来自混杂的数据增量。

In [ ]:
# 四个配置：是否加新模块 × 数据量(1x / 2x)。准确率为（仿真）实测
exp = pd.DataFrame({
    'module':   [False, True,  False, True],
    'data':     ['1x',  '1x',  '2x',  '2x'],
    'accuracy': [0.840, 0.850, 0.862, 0.871],
})
print(exp.to_string(index=False))

def acc(module, data):
    m = (exp['module'] == module) & (exp['data'] == data)
    return exp.loc[m, 'accuracy'].iloc[0]

# 论文的不诚实对比：基线(无模块,1x数据) vs 新方法(有模块,2x数据)
naive_gap = acc(True, '2x') - acc(False, '1x')
# 受控分解：固定数据=1x，只看模块的纯效应
module_effect = acc(True, '1x') - acc(False, '1x')
# 固定无模块，只看数据的纯效应
data_effect = acc(False, '2x') - acc(False, '1x')
print(f'\n论文宣称的总提升(基线1x vs 新方法2x) = {naive_gap:+.3f}')
print(f'  其中 模块的纯效应(控住数据=1x)    = {module_effect:+.3f}')
print(f'  其中 数据的纯效应(控住无模块)      = {data_effect:+.3f}')
assert naive_gap > module_effect, '宣称的提升被数据混杂放大了'
assert data_effect > module_effect, '本例中数据的功劳其实大于模块'
print('✅ 拆穿：宣称的 +0.031 里，模块只贡献 +0.010，数据(混杂)贡献更多。')
print('   读消融/对比表时，永远问：两个对比配置之间，是不是只差了那一个变量？')

## 5 · worked：可信度打分器

把『一个结果可不可信』拆成 7 项可勾选的检查，量化打分并指出短板。读任何论文结果都可以过一遍这张表。

In [ ]:
def credibility_score(checks: dict):
    '''checks: 7 项布尔检查。返回 (得分, 满分, 缺失项)。'''
    score = sum(bool(v) for v in checks.values())
    missing = [k for k, v in checks.items() if not v]
    return score, len(checks), missing

# 结果一：刷榜风格（SOTA 但不严谨）
result_sota = dict(有对照=True, 有消融=False, 多种子=False, 报方差=False,
                   强基线=False, 不过度声明=False, 可复现公开=True)
# 结果二：朴素但扎实
result_solid = dict(有对照=True, 有消融=True, 多种子=True, 报方差=True,
                    强基线=True, 不过度声明=True, 可复现公开=True)

for name, r in [('SOTA但不严谨', result_sota), ('朴素但扎实', result_solid)]:
    s, total, miss = credibility_score(r)
    print(f'{name:14s} 可信度 {s}/{total}  短板: {miss if miss else "无"}')
s1, _, _ = credibility_score(result_sota)
s2, _, _ = credibility_score(result_solid)
assert s2 > s1, '扎实的结果应比刷榜结果可信度更高'
print('✅ 可信度不看是否 SOTA，看证据是否齐全。朴素而扎实 > 漂亮而空洞。')

---
## ✏️ 练习 1：从主张里提取「待验证的证据需求」

给定一个主张字符串和它的「范围标签」（声称适用的任务数 `claimed_tasks` 与实际验证的任务数 `tested_tasks`），
实现 `overclaim_gap(claimed, tested)`：返回「过度声明缺口」= 声称但未验证的任务数。缺口 > 0 即过度声明。

In [ ]:
def overclaim_gap(claimed_tasks, tested_tasks):
    # TODO: 返回 max(0, claimed_tasks - tested_tasks)
    #       即：声称适用 claimed 个任务，但只在 tested 个上做了实验，
    #       缺口 = 没被证据覆盖的任务数。
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert overclaim_gap(5, 2) == 3, '声称5个任务只验证2个 -> 缺口3'
assert overclaim_gap(2, 2) == 0, '声称=验证 -> 无过度声明'
assert overclaim_gap(1, 3) == 0, '验证多于声称 -> 不算过度声明（缺口取0）'
print('✅ 练习 1 通过：能量化主张范围与证据范围的缺口（过度声明检测）')

## ✏️ 练习 2：定位复现差异的来源

你复现论文报告的 `accuracy=0.92`，但只算出 `0.88`。差异来自哪里？常见元凶：
(a) 用错了指标定义；(b) 用了不同的数据子集。

给定论文预测 `paper_df` 和你的预测 `my_df`，实现 `diagnose(paper_df, my_df)`：
若两者**样本数不同** → 返回 `'data_mismatch'`；若样本数相同但**预测不同** → 返回 `'pred_mismatch'`；完全相同 → `'match'`。

In [ ]:
def diagnose(paper_df, my_df):
    # TODO: 比较两个 DataFrame（列 y_true,y_pred）。
    #   1) 行数不同 -> 'data_mismatch'
    #   2) 行数同但 y_pred 不完全相同 -> 'pred_mismatch'
    #   3) 完全一致 -> 'match'
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
base = pd.DataFrame({'y_true':[0,1,1,0], 'y_pred':[0,1,1,0]})
fewer = pd.DataFrame({'y_true':[0,1], 'y_pred':[0,1]})              # 数据少了
diffpred = pd.DataFrame({'y_true':[0,1,1,0], 'y_pred':[0,0,1,0]})   # 预测不同
assert diagnose(base, fewer) == 'data_mismatch'
assert diagnose(base, diffpred) == 'pred_mismatch'
assert diagnose(base, base.copy()) == 'match'
print('✅ 练习 2 通过：能区分复现差异是『数据不同』还是『预测/实现不同』')

## ✏️ 练习 3：消融的可信度——掉的分显著吗？

消融显示「去掉组件 A 掉了 `drop` 分」。但如果运行间标准差是 `std`，这点掉分可能在噪声内。

实现 `ablation_is_significant(drop, std, k=2.0)`：当掉分**大于** `k` 倍标准差时才算可信（返回 True）。
（这是模块 03 严格统计检验的简化直觉版：效应要明显超过噪声。）

In [ ]:
def ablation_is_significant(drop, std, k=2.0):
    # TODO: 返回 drop > k * std
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert ablation_is_significant(drop=0.030, std=0.005) == True,  '0.030 >> 2*0.005'
assert ablation_is_significant(drop=0.008, std=0.010) == False, '0.008 < 2*0.010 噪声内'
assert ablation_is_significant(drop=0.025, std=0.010) == True
print('✅ 练习 3 通过：能判断消融掉的分是真效应还是落在运行噪声里')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def overclaim_gap(claimed_tasks, tested_tasks):
    return max(0, claimed_tasks - tested_tasks)

In [ ]:
# 练习 2 参考答案
def diagnose(paper_df, my_df):
    if len(paper_df) != len(my_df):
        return 'data_mismatch'
    if not np.array_equal(paper_df['y_pred'].values, my_df['y_pred'].values):
        return 'pred_mismatch'
    return 'match'

In [ ]:
# 练习 3 参考答案
def ablation_is_significant(drop, std, k=2.0):
    return drop > k * std

---
## 🧪 真实数据胶囊：复现一个真实论文报告的指标关系

我们用一个**真实、稳定**的事实做复现练习：在类别**不平衡**的多分类里，`micro-F1` 恒等于 `accuracy`，而 `macro-F1` 可显著不同。这是 scikit-learn 文档与无数论文都依赖的关系。

下面尝试用 sklearn 验证（若装了），否则**回退到我们自己的实现**——两条路都应得出同一结论，这本身就是一次『独立复现』。

In [ ]:
yt = np.array([0]*900 + [1]*80 + [2]*20)
yp = yt.copy(); yp[900:] = 0     # 两个小类全预测错

# 路线 A：用我们模块 2 的实现
mine_micro = micro_f1(yt, yp)
mine_macro = macro_f1(yt, yp)
mine_acc   = (yp == yt).mean()

# 路线 B：尝试用 sklearn 独立复现（联网/已装则用，否则回退）
try:
    from sklearn.metrics import f1_score, accuracy_score
    sk_micro = f1_score(yt, yp, average='micro')
    sk_macro = f1_score(yt, yp, average='macro')
    sk_acc   = accuracy_score(yt, yp)
    src = 'sklearn'
except Exception:
    sk_micro, sk_macro, sk_acc = mine_micro, mine_macro, mine_acc   # 回退到内置实现
    src = '内置实现(回退)'

print(f'指标来源: {src}')
print(f'micro-F1 = {sk_micro:.4f}   accuracy = {sk_acc:.4f}   (应相等)')
print(f'macro-F1 = {sk_macro:.4f}   (应明显更低)')
# 复现核对：micro-F1 == accuracy 这个『论文级事实』
assert abs(sk_micro - sk_acc) < 1e-9, '多分类下 micro-F1 必等于 accuracy'
assert sk_macro < sk_acc - 0.3, '不平衡时 macro 应远低于 accuracy'
# 两条独立路线一致 = 复现成功
assert abs(mine_macro - sk_macro) < 1e-9 and abs(mine_micro - sk_micro) < 1e-9
print('✅ 两条独立实现得出同一结论 —— 这就是一次成功的【独立复现】。')

**🧪 胶囊练习**：实现 `weighted_f1(y_true, y_pred)`：按各类**样本数加权**平均各类 F1（第三种常见 F1）。验证它落在 macro 与 micro 之间（不平衡时通常如此）。

In [ ]:
def weighted_f1(y_true, y_pred):
    # TODO: 对每个类算 F1，用该类的真实样本数占比加权求和
    #   classes = np.unique(y_true)
    #   weight_c = (y_true == c).sum() / len(y_true)
    #   return sum(weight_c * f1_c)
    raise NotImplementedError

In [ ]:
# 自测
wf = weighted_f1(yt, yp)
print(f'weighted-F1 = {wf:.4f}')
# 大类(类0)占 90% 且全对，所以 weighted 应被大类拉高，接近 micro
assert mine_macro <= wf + 1e-9, 'weighted 通常 >= macro（大类权重高）'
assert wf <= 1.0
print('✅ 胶囊练习通过：weighted-F1 实现正确')

In [ ]:
# 📖 胶囊参考答案
def weighted_f1(y_true, y_pred):
    classes = np.unique(y_true)
    total = len(y_true)
    out = 0.0
    for cls in classes:
        w = (y_true == cls).sum() / total
        out += w * precision_recall_f1(y_true, y_pred, cls)[2]
    return out

### 小结
- **三遍读法**：尽早判断去留，第三遍『虚拟复现』把读懂推进到能批判/能复现。
- **主张-证据对**：把论文拆成断言+凭证，找出没有证据的主张（过度声明红旗）。
- **读消融表**：算每个组件的边际贡献、揪出名不副实的组件、警惕把数据/算力的功劳记到方法头上（混杂）。
- **复现的单位是一个具体数字**；指标的『确切定义』(macro vs micro) 是复现对不上的头号元凶。
- **可信度打分**：看证据是否齐全（对照/消融/方差/强基线/不过度声明/可复现），而非是否 SOTA。

下一站：**模块 02 · 实验设计与消融纪律** —— 学会读别人的实验后，自己怎么设计能下结论的实验。